# 00 — Descargar e inspeccionar MIMIC-IV Demo v2.2

Descarga la muestra pública oficial de 100 pacientes, valida cada fichero mediante SHA-256 y consulta los CSV comprimidos directamente con DuckDB. No requiere ni solicita credenciales de PhysioNet. El directorio `data/` está excluido de Git.

La muestra sirve para probar código, tablas y relaciones. No tiene tamaño suficiente para entrenar o validar el modelo ni para estimar prevalencias.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from mimic_sepsis.demo import DEMO_VERSION, download_demo

DATA_DIR = PROJECT_ROOT / 'data' / 'mimic-iv-demo' / DEMO_VERSION
files = download_demo(DATA_DIR)
print(f'{len(files)} ficheros verificados en {DATA_DIR}')

In [ ]:
import duckdb

connection = duckdb.connect()
patients_path = str(DATA_DIR / 'hosp' 
                    / 'patients.csv.gz')
admissions_path = str(DATA_DIR / 'hosp' / 'admissions.csv.gz')
icustays_path = str(DATA_DIR / 'icu' / 'icustays.csv.gz')

In [ ]:
summary = connection.execute(
    """
    WITH patients AS (SELECT * FROM read_csv_auto(?)),
         admissions AS (SELECT * FROM read_csv_auto(?)),
         icustays AS (SELECT * FROM read_csv_auto(?))
    SELECT
        (SELECT COUNT(*) FROM patients) AS patients,
        (SELECT COUNT(*) FROM admissions) AS admissions,
        (SELECT COUNT(*) FROM icustays) AS icu_stays,
        (SELECT COUNT(DISTINCT subject_id) FROM icustays) AS icu_patients
    """,
    [patients_path, admissions_path, icustays_path],
).df()
summary

In [ ]:
tables = []
for domain in ('hosp', 'icu'):
    for path in sorted((DATA_DIR / domain).glob('*.csv.gz')):
        count = connection.execute(
            'SELECT COUNT(*) FROM read_csv_auto(?)', [str(path)]
        ).fetchone()[0]
        tables.append({'domain': domain, 'table': path.name.removesuffix('.csv.gz'), 'rows': count})

import pandas as pd
pd.DataFrame(tables).sort_values(['domain', 'table']).reset_index(drop=True)

## Siguiente comprobación

Con la estructura validada, el siguiente notebook implementará una cohorte adulta sobre estos ficheros. Las consultas clínicas reutilizables se moverán a `sql/` y deberán funcionar también contra el backend completo con adaptaciones explícitas del dialecto.

In [ ]:
connection.close()